In [4]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
import math

W, H = 900, 420
N_FRAMES = 40
DURATION_MS = 90
FREQ_MULT = 4.0
def get_font(size: int):
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ]
    for p in candidates:
        if Path(p).exists():
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

font_title = get_font(28)
font_label = get_font(22)
font_small = get_font(18)

BG = (250, 250, 250)
BLACK = (20, 20, 20)
GRAY = (120, 120, 120)
BLUE = (50, 120, 220)
GREEN = (30, 150, 80)
RED = (220, 60, 60)

frames = []

for i in range(N_FRAMES):
    t = 2 * math.pi * i / N_FRAMES
    img = Image.new("RGB", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Title
    d.text((W // 2 - 160, 16), "Translation vs Stretch", fill=BLACK, font=font_title)

    # Separator
    d.line((W // 2, 70, W // 2, H - 30), fill=(180, 180, 180), width=2)

    # ===== Left panel: Translation =====
    left_x0, left_x1 = 40, W // 2 - 30
    d.text((120, 70), "Translation", fill=BLACK, font=font_label)

    surf_y = 315
    d.rectangle((left_x0 + 30, surf_y, left_x1 - 30, surf_y + 16), fill=(170, 170, 170), outline=GRAY)
    d.text((left_x0 + 60, surf_y + 24), "Surface", fill=GRAY, font=font_small)

    tip_top = 110
    tip_center_x = (left_x0 + left_x1) // 2
    tip_w = 120
    d.polygon([
        (tip_center_x - tip_w // 2, tip_top),
        (tip_center_x + tip_w // 2, tip_top),
        (tip_center_x + 18, 220),
        (tip_center_x - 18, 220),
    ], fill=(205, 205, 205), outline=GRAY)
    d.text((tip_center_x - 18, 82), "Tip", fill=GRAY, font=font_small)

    z_offset = 18 * math.sin(t)
    mol_y = int(270 - z_offset)
    bond_len = 40
    r = 14
    x1 = int(tip_center_x - bond_len // 2)
    x2 = int(tip_center_x + bond_len // 2)

    d.line((x1, mol_y, x2, mol_y), fill=BLACK, width=3)
    d.ellipse((x1 - r, mol_y - r, x1 + r, mol_y + r), fill=BLUE, outline=BLACK)
    d.ellipse((x2 - r, mol_y - r, x2 + r, mol_y + r), fill=BLUE, outline=BLACK)

    arrow_x = left_x0 + 95
    d.line((arrow_x, surf_y - 4, arrow_x, mol_y), fill=RED, width=3)
    d.polygon([(arrow_x, mol_y), (arrow_x - 6, mol_y + 12), (arrow_x + 6, mol_y + 12)], fill=RED)
    d.polygon([(arrow_x, surf_y - 4), (arrow_x - 6, surf_y - 16), (arrow_x + 6, surf_y - 16)], fill=RED)
    d.text((arrow_x - 18, (surf_y + mol_y) // 2 - 10), "d", fill=RED, font=font_small)

    d.text((72, H - 55), "Whole molecule moves up/down", fill=BLACK, font=font_small)

    # ===== Right panel: Stretch =====
    right_x0, right_x1 = W // 2 + 30, W - 40
    d.text((right_x0 + 90, 70), "Stretch", fill=BLACK, font=font_label)

    d.rectangle((right_x0 + 40, surf_y, right_x1 - 40, surf_y + 16), fill=(170, 170, 170), outline=GRAY)
    d.text((right_x0 + 78, surf_y + 24), "Center fixed", fill=GRAY, font=font_small)

    tip_center_x2 = (right_x0 + right_x1) // 2
    d.polygon([
        (tip_center_x2 - tip_w // 2, tip_top),
        (tip_center_x2 + tip_w // 2, tip_top),
        (tip_center_x2 + 18, 220),
        (tip_center_x2 - 18, 220),
    ], fill=(205, 205, 205), outline=GRAY)
    bond_len2 = 30 + 14 * math.sin(FREQ_MULT * t)
    mol_y2 = 255
    x1b = tip_center_x2 - bond_len2 / 2
    x2b = tip_center_x2 + bond_len2 / 2

    d.line((x1b, mol_y2, x2b, mol_y2), fill=BLACK, width=3)
    d.ellipse((x1b - r, mol_y2 - r, x1b + r, mol_y2 + r), fill=GREEN, outline=BLACK)
    d.ellipse((x2b - r, mol_y2 - r, x2b + r, mol_y2 + r), fill=GREEN, outline=BLACK)

    y_arrow = 205
    d.line((x1b, y_arrow, x2b, y_arrow), fill=RED, width=3)
    d.polygon([(x1b, y_arrow), (x1b + 12, y_arrow - 6), (x1b + 12, y_arrow + 6)], fill=RED)
    d.polygon([(x2b, y_arrow), (x2b - 12, y_arrow - 6), (x2b - 12, y_arrow + 6)], fill=RED)
    d.text((tip_center_x2 - 10, y_arrow - 35), "r", fill=RED, font=font_small)

    d.text((right_x0 + 62, H - 55), "Bond length changes only", fill=BLACK, font=font_small)

    frames.append(img)

gif_path = Path("translation_vs_stretch_clean.gif")
frames[0].save(
    str(gif_path),
    save_all=True,
    append_images=frames[1:],
    duration=DURATION_MS,
    loop=0,
)

print("Saved GIF:", gif_path)

Saved GIF: translation_vs_stretch_clean.gif
